# 11 - Train one run (Kaggle or Colab, T4)

Trains the predictor with one config from `configs/train/` and saves checkpoints to the private model repo `Tsomaros/llm-movies-runs`.
**Rerunning with the same config resumes** from the run's `last` checkpoint on the Hub, so a session that ends loses at most the steps since the last save (`save_every_steps`).

- **Kaggle:** Settings > Accelerator > GPU T4, Internet on. Add-ons > Secrets: `HF_TOKEN` (write access) and `WANDB_API_KEY`, both attached to this notebook. Run the Setup cells, then the **Kaggle** secrets cell, not the Colab one.
- **Colab:** Runtime > Change runtime type > T4 GPU. Colab Secrets (key icon, notebook access on): `HF_TOKEN` and `WANDB_API_KEY`. Run the Setup cells, then the **Colab** secrets cell, not the Kaggle one.

Both secrets are **required**: the secrets cell fails if one is missing. Values are never printed.

## Setup

Run these top to bottom. The sync installs the `train` extra and CUDA 13.2 PyTorch (`cu132`) and takes a few minutes.
**Before the sync, check the GPU software** in a scratch cell with `!nvidia-smi`: "CUDA Version" must be **13.2 or higher**. Colab's T4 passed this in Task 10; on Kaggle it is unverified, so if it shows less, use Colab.

In [ ]:
# clone the repo the first time, update it after that
!test -d /content/LLM_Movies_Model/.git && git -C /content/LLM_Movies_Model pull || git clone https://github.com/Tsomaros/LLM_Movies_Model /content/LLM_Movies_Model
%cd /content/LLM_Movies_Model

In [ ]:
!pip install -q uv
!uv sync --frozen --extra train --extra cu132

## Secrets: Colab only

Skip this cell on Kaggle.

In [ ]:
import os

from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")

## Secrets: Kaggle only

Skip this cell on Colab.

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")

## Environment

`UV_FROZEN` stops `uv run` from re-resolving the environment.

In [ ]:
import os

os.environ["UV_FROZEN"] = "1"

## Train

Pick the config: from the dropdown on Colab, or by editing the `CONFIG` line on Kaggle, which has no form. `base` is the shared starting point, `smoke` is the tiny Task 11 Step 4 check, and E1 to E5 are the Task 12 experiments. Each has its own `run_name`, so its checkpoints and W&B run are separate.
To resume after a crash or the end of a session, run the whole notebook again with the same config. **Never edit a config while its run is unfinished**: the saved step count would no longer match.

In [ ]:
CONFIG = "configs/train/base.yaml"  #@param ["configs/train/base.yaml", "configs/train/smoke.yaml", "configs/train/e1_small.yaml", "configs/train/e1_large.yaml", "configs/train/e2_anon.yaml", "configs/train/e3_notext.yaml", "configs/train/e4_noleak.yaml", "configs/train/e5_nosource.yaml"]
!uv run python -m llm_movies.model.train --config {CONFIG}